In [18]:
#!/usr/bin/env python3
"""
Projected iterated commutators in OpenFermion (symbolic; no matrices).

System: N qubits (sites 0..N-1)
Hamiltonian (open boundary): H = J * Σ_{i=0}^{N-2} (X_i X_{i+1} + Y_i Y_{i+1})
Seed observable: O0 = Z0
Iterates: ad_H^n(O0) = [H, [H, ... [H, O0] ... ]]

Projection: π_m onto <=m-body operators induced by a product reference state σ0,
specified by a 3N Bloch vector:
  bloch = [<X0>,<Y0>,<Z0>, <X1>,<Y1>,<Z1>, ..., <X_{N-1}>,<Y_{N-1}>,<Z_{N-1}>]

This file computes:
  - Unprojected iterated commutators ad_H^n(O0)
  - Projected commutators π_m(ad_H^n(O0))
  - Optionally evaluates Bloch symbols to numbers (e.g., all x=y=0, z=1)

Requires:
  pip install openfermion sympy
"""

import sympy as sp
from itertools import combinations
from openfermion.ops import QubitOperator
from openfermion.utils.commutators import commutator


# -----------------------------
# Basic utilities
# -----------------------------

def simplify_qubit_operator(op: QubitOperator) -> QubitOperator:
    """Sympy-simplify coefficients term-by-term and drop exact zeros."""
    out = QubitOperator()
    for term, coeff in op.terms.items():
        c = sp.simplify(sp.sympify(coeff))
        if c != 0:
            out += QubitOperator(term, c)
    return out


def substitute_coeffs(op: QubitOperator, subs: dict) -> QubitOperator:
    """Substitute SymPy symbols inside coefficients of a QubitOperator."""
    out = QubitOperator()
    for term, coeff in op.terms.items():
        newc = sp.simplify(sp.sympify(coeff).subs(subs))
        if newc != 0:
            out += QubitOperator(term, newc)
    return simplify_qubit_operator(out)


# -----------------------------
# Hamiltonian and commutators
# -----------------------------

def xy_hamiltonian_open_boundary(n_sites: int, coupling) -> QubitOperator:
    """
    H = coupling * Σ_{i=0}^{n_sites-2} (X_i X_{i+1} + Y_i Y_{i+1})
    """
    H = QubitOperator()
    for i in range(n_sites - 1):
        H += coupling * QubitOperator(f"X{i} X{i+1}")
        H += coupling * QubitOperator(f"Y{i} Y{i+1}")
    return simplify_qubit_operator(H)


def iterated_commutators_real_generator(H: QubitOperator, O0: QubitOperator, n_max: int):
    """
    Returns [O0, L(O0), L^2(O0), ...] where L(O) = -i [H,O].
    This matches Phoenix's real_only commutator convention.
    """
    seq = [simplify_qubit_operator(O0)]
    cur = seq[0]
    for _ in range(n_max):
        cur = simplify_qubit_operator((-sp.I) * commutator(H, cur))
        seq.append(cur)
    return seq



# -----------------------------
# π_m projection (product reference state)
# -----------------------------

_pauli_index = {'X': 0, 'Y': 1, 'Z': 2}


def mu_from_bloch(bloch, site: int, pauli: str):
    """Return μ = <pauli_site> from a length-3N bloch vector."""
    return bloch[3 * site + _pauli_index[pauli]]


def _term_to_site_map(term):
    """term: tuple like ((i,'X'), (j,'Z'), ...) -> dict {i:'X', j:'Z'}"""
    return {i: p for (i, p) in term}


def _site_map_to_term_tuple(site_map):
    """dict {i:'X', j:'Z'} -> sorted tuple ((i,'X'), (j,'Z'), ...)"""
    if not site_map:
        return tuple()
    return tuple(sorted(site_map.items(), key=lambda x: x[0]))


def project_pauli_string_pi_m(term, coeff, m: int, bloch):
    """
    Project a single Pauli string term onto <=m-body operators using:
      σ = δσ + μ I, truncate to <=m deltas, then expand δσ = σ - μ I.

    This is the inclusion–exclusion structure you want, e.g. for m=2:
      ABC -> aBC+bAC+cAB -abC -acB -bcA + abc I.
    """
    site_map = _term_to_site_map(term)
    active_sites = sorted(site_map.keys())
    k = len(active_sites)

    out = QubitOperator()

    # Choose which sites carry δσ (r of them); keep only r<=m
    for r in range(0, min(m, k) + 1):
        for delta_subset in combinations(active_sites, r):
            delta_subset = set(delta_subset)

            branch_coeff = sp.sympify(coeff)
            delta_map = {}  # sites -> Pauli label (for δσ factors)

            # For non-delta sites: contribute μ * I
            for s in active_sites:
                p = site_map[s]
                if s in delta_subset:
                    delta_map[s] = p
                else:
                    branch_coeff *= mu_from_bloch(bloch, s, p)

            # Expand each δσ = σ - μ I (no further truncation needed since r<=m)
            delta_sites_sorted = sorted(delta_map.keys())
            for t in range(0, len(delta_sites_sorted) + 1):
                for keep_subset in combinations(delta_sites_sorted, t):
                    keep_subset = set(keep_subset)

                    final_coeff = branch_coeff
                    final_site_map = {}

                    for s in delta_sites_sorted:
                        p = delta_map[s]
                        if s in keep_subset:
                            final_site_map[s] = p
                        else:
                            final_coeff *= (-mu_from_bloch(bloch, s, p))

                    final_term = _site_map_to_term_tuple(final_site_map)
                    out += QubitOperator(final_term, final_coeff)

    return out


def pi_m(op: QubitOperator, m: int, bloch) -> QubitOperator:
    """Mean-field / product-state covariance projection π_m onto <=m-body operators."""
    out = QubitOperator()
    for term, coeff in op.terms.items():
        out += project_pauli_string_pi_m(term, coeff, m, bloch)
    return simplify_qubit_operator(out)


# -----------------------------
# Bloch presets
# -----------------------------

def make_symbolic_bloch(N: int):
    """
    Returns (bloch_syms, sym_dict) where:
      bloch_syms is length 3N list [x0,y0,z0,x1,y1,z1,...]
      sym_dict maps ('x',i) -> Symbol('x{i}'), etc.
    """
    bloch = []
    sym_dict = {}
    for i in range(N):
        xi, yi, zi = sp.symbols(f"x{i} y{i} z{i}")
        bloch += [xi, yi, zi]
        sym_dict[('x', i)] = xi
        sym_dict[('y', i)] = yi
        sym_dict[('z', i)] = zi
    return bloch, sym_dict


def preset_subs_all_xy_zero_all_z_one(N: int):
    """All sites: x_i=0, y_i=0, z_i=1."""
    subs = {}
    for i in range(N):
        subs[sp.Symbol(f"x{i}")] = 0
        subs[sp.Symbol(f"y{i}")] = 0
        subs[sp.Symbol(f"z{i}")] = 1
    return subs


def preset_subs_infinite_temperature(N: int):
    """All sites maximally mixed: x_i=y_i=z_i=0."""
    subs = {}
    for i in range(N):
        subs[sp.Symbol(f"x{i}")] = .0
        subs[sp.Symbol(f"y{i}")] = .0
        subs[sp.Symbol(f"z{i}")] = 1.
    return subs


def preset_subs_single_site_polarized(N: int, site: int = 0, axis: str = 'Z', value=1):
    """
    All sites maximally mixed except one site polarized along given axis.
    Example: site=0, axis='Z', value=1 => z0=1, others 0.
    """
    subs = preset_subs_all_xy_zero_all_z_one(N)
    axis = axis.upper()
    if axis not in ('X', 'Y', 'Z'):
        raise ValueError("axis must be one of 'X','Y','Z'")
    key = {'X': 'x', 'Y': 'y', 'Z': 'z'}[axis]
    subs[sp.Symbol(f"{key}{site}")] = value
    return subs


# -----------------------------
# Main (drop-in)
# -----------------------------
def main():
    # ----- parameters -----
    N = 4
    ell = 9
    m = 2

    J = sp.Symbol("J")  # symbolic coupling

    # ----- build H and seed -----
    H = xy_hamiltonian_open_boundary(N, J)
    O0 = .5 * QubitOperator("Z0")

    # ----- compute commutators -----
    ads = iterated_commutators(H, O0, ell)

    # ----- build symbolic bloch -----
    bloch_syms, _ = make_symbolic_bloch(N)

    # ----- project -----
    ads_proj = [pi_m(On, m=m, bloch=bloch_syms) for On in ads]

    # ----- choose evaluation preset (optional) -----
    # Example requested: all x=y=0 and all z=1
    subs = preset_subs_all_xy_zero_all_z_one(N)

    ads_proj_eval = [substitute_coeffs(Pn, subs) for Pn in ads_proj]

    # ----- prints -----
    print("H =", H)
    print("O0 =", O0)

    for n, (On, Pn, Pn_eval) in enumerate(zip(ads, ads_proj, ads_proj_eval)):
        print(f"\n==================== n={n} ====================")
        print(f"ad_H^{n}(O0) =")
        print(On)
        print(f"\npi_{m}(ad_H^{n}(O0)) =")
        print(Pn)
        print(f"\npi_{m}(ad_H^{n}(O0)) evaluated (preset) =")
        print(Pn_eval)


if __name__ == "__main__":
    main()


H = 1.0*J [X0 X1] +
1.0*J [Y0 Y1] +
1.0*J [X1 X2] +
1.0*J [Y1 Y2] +
1.0*J [X2 X3] +
1.0*J [Y2 Y3]
O0 = 0.5 [Z0]

==================== n=0 ====================
ad_H^0(O0) =
0.500000000000000 [Z0]

pi_2(ad_H^0(O0)) =
0.500000000000000 [Z0]

pi_2(ad_H^0(O0)) evaluated (preset) =
0.500000000000000 [Z0]

==================== n=1 ====================
ad_H^1(O0) =
1.0*J [X0 Y1] +
-1.0*J [Y0 X1]

pi_2(ad_H^1(O0)) =
1.0*J [X0 Y1] +
-1.0*J [Y0 X1]

pi_2(ad_H^1(O0)) evaluated (preset) =
1.0*J [X0 Y1] +
-1.0*J [Y0 X1]

==================== n=2 ====================
ad_H^2(O0) =
2.0*J**2 [X0 Z1 X2] +
2.0*J**2 [Y0 Z1 Y2] +
-4.0*J**2 [Z0] +
4.0*J**2 [Z1]

pi_2(ad_H^2(O0)) =
2.0*J**2*z1*(x0*x2 + y0*y2) [] +
-2.0*J**2*x2*z1 [X0] +
2.0*J**2*x2 [X0 Z1] +
2.0*J**2*z1 [X0 X2] +
-2.0*J**2*y2*z1 [Y0] +
2.0*J**2*y2 [Y0 Z1] +
2.0*J**2*z1 [Y0 Y2] +
-4.0*J**2 [Z0] +
J**2*(-2.0*x0*x2 - 2.0*y0*y2 + 4.0) [Z1] +
2.0*J**2*x0 [Z1 X2] +
2.0*J**2*y0 [Z1 Y2] +
-2.0*J**2*x0*z1 [X2] +
-2.0*J**2*y0*z1 [Y2]

pi_2(ad_H^2(O0)) 